# Objectives

- Splitting cleaned dataset into train and test sets.
- Encode categorical and numerical features in preparation for future modeling

# Imports

In [1]:
import pandas as pd
import math
import random
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
import time
import plotly.express as px
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['legend.fontsize'] = 10
import joblib

# Load clean dataset

In [2]:
df = pd.read_csv("../data/processed/diabetes_clean.csv")
df.head()

,race,gender,age,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted,admission_type,discharge_disposition,admission_source
0,Caucasian,Female,[0-10),1,Pediatrics-Endocrinology,41,0,1,0,0,...,No,No,No,No,No,No,NO,NaN,Not Mapped,Physician Referral
1,Caucasian,Female,[10-20),3,Unknown,59,0,18,0,0,...,No,No,Up,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,AfricanAmerican,Female,[20-30),2,Unknown,11,5,13,2,0,...,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,Caucasian,Male,[30-40),2,Unknown,44,1,16,0,0,...,No,No,Up,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,Caucasian,Male,[40-50),1,Unknown,51,0,8,0,0,...,No,No,Steady,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room


# Data Splitting

In [3]:
X = df.drop('readmitted', axis = 1)
Y = df['readmitted']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.25, random_state=42, stratify=Y
)

# Feature classification

In [5]:
numerical_features = [
    "time_in_hospital",
    "num_procedures",
    "num_lab_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

In [6]:
categorical_features = [
    col for col in X.columns
    if col not in numerical_features
]

# Encoding

## One-hot encoder

In [7]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [8]:
encoder.fit(x_train[categorical_features])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [9]:
x_train_cat = encoder.transform(x_train[categorical_features])

x_test_cat = encoder.transform(x_test[categorical_features])

## Standar Scaler

In [10]:
scaler = StandardScaler()

scaler.fit(x_train[numerical_features])

StandardScaler()

In [11]:
x_train_num = scaler.transform(x_train[numerical_features])

x_test_num = scaler.transform(x_test[numerical_features])

# Convert to Dataframes

In [12]:
x_train_cat = pd.DataFrame(
    x_train_cat,
    columns=encoder.get_feature_names_out(categorical_features),
    index=x_train.index
)

In [13]:
x_test_cat = pd.DataFrame(
    x_test_cat,
    columns=encoder.get_feature_names_out(categorical_features),
    index=x_test.index
)

In [14]:
x_train_num = pd.DataFrame(
    x_train_num,
    columns=numerical_features,
    index=x_train.index
)

In [15]:
x_test_num = pd.DataFrame(
    x_test_num,
    columns=numerical_features,
    index=x_test.index
)

In [16]:
x_train_processed = pd.concat(
    [x_train_num, x_train_cat],
    axis=1
)

x_test_processed = pd.concat(
    [x_test_num, x_test_cat],
    axis=1
)

In [17]:
x_test_processed.shape

(25441, 469)

In [18]:
x_train_processed.shape

(76322, 469)

In [19]:
x_train_processed.isna().sum().sum()

np.int64(0)

# Saving

In [20]:
x_train_processed.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

x_test_processed.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [21]:

joblib.dump(
    encoder,
    "../models/onehot_encoder.pkl"
)

joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

['../models/standard_scaler.pkl']